# Phase 5: Stockfish CPL Baseline
Đây là notebook đánh giá hiệu năng XGBoost trên tập features 113 cột (Stockfish CPL + Tabular).

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
# Load data
df = pl.read_parquet('../data/features/sample_30k_features.parquet')
print(f'Shape: {df.shape}')

y = df['ModelBand'].to_numpy()
features = [c for c in df.columns if c != 'ModelBand']
X = df.select(features).to_numpy()

engine_cols = ['avg_cpl', 'blunder_count', 'mistake_count', 'inaccuracy_count', 'max_cpl', 'cpl_std']
tabular_features = [c for c in features if c not in engine_cols]
X_tabular = df.select(tabular_features).to_numpy()


In [ ]:
def evaluate_model(X_data, y_target, feat_names, title):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accs, f1s = [], []
    oof_preds = np.zeros_like(y_target)
    importances = np.zeros(X_data.shape[1])
    
    params = {
        'objective': 'multi:softmax',
        'num_class': 5,
        'eval_metric': 'mlogloss',
        'max_depth': 6,
        'learning_rate': 0.1,
        'n_estimators': 100,
        'random_state': 42,
        'n_jobs': -1
    }
    
    for train_idx, val_idx in skf.split(X_data, y_target):
        X_tr, y_tr = X_data[train_idx], y_target[train_idx]
        X_va, y_va = X_data[val_idx], y_target[val_idx]
        
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        
        preds = clf.predict(X_va)
        oof_preds[val_idx] = preds
        
        accs.append(accuracy_score(y_va, preds))
        f1s.append(f1_score(y_va, preds, average='macro'))
        importances += clf.feature_importances_ / skf.n_splits
        
    print(f'=== {title} ===')
    print(f'Accuracy: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%')
    print(f'Macro F1: {np.mean(f1s)*100:.2f}% ± {np.std(f1s)*100:.2f}%')
    
    return oof_preds, importances

print('TABULAR ONLY (Ablation Study):')
preds_tab, imp_tab = evaluate_model(X_tabular, y, tabular_features, 'Tabular Only')

print('\nSTOCKFISH + TABULAR (Full Features):')
preds_all, imp_all = evaluate_model(X, y, features, 'Stockfish + Tabular')


In [ ]:
cm = confusion_matrix(y, preds_all)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Beginner','Intermediate','Advanced','Expert','Master'],
            yticklabels=['Beginner','Intermediate','Advanced','Expert','Master'])
plt.title('Confusion Matrix - Stockfish CPL XGBoost')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
top_idx = np.argsort(imp_all)[-15:]
plt.figure(figsize=(10,6))
plt.barh(np.array(features)[top_idx], imp_all[top_idx])
plt.title('Top 15 Feature Importances (Stockfish + Tabular)')
plt.show()
